<a href="https://colab.research.google.com/github/BreakTheAlgorithm/MLforALL/blob/main/book_ch16_end_to_end_projects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🏗️ Chapter 16 — End-to-End Machine Learning Projects</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 90 mins | Level: Advanced</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Follow a structured end-to-end ML project workflow
- Frame a business problem as an ML problem
- Build a full pipeline from raw data to model evaluation
- Document assumptions, limitations, and next steps
- Interpret model outputs for non-technical stakeholders

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection  import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing    import StandardScaler, LabelEncoder
from sklearn.impute            import SimpleImputer
from sklearn.pipeline         import Pipeline
from sklearn.compose          import ColumnTransformer
from sklearn.ensemble         import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.metrics          import (classification_report, confusion_matrix,
                                       roc_auc_score, roc_curve)

%matplotlib inline
np.random.seed(42)

## 📖 Step 1 — Problem Framing

**Business Problem:** A Bangalore-based fintech startup wants to predict whether a customer will default on a personal loan within 6 months. Reduces credit risk exposure by ₹5-20 crore per quarter.

**ML Problem:** Binary classification — predict `default` (1=will default, 0=will not default).

**Success Metric:** Recall (Sensitivity) > 0.80 at precision ≥ 0.60 — we prefer catching real defaulters over false negatives.

**Constraints:** Model must be interpretable for RBI compliance. Inference < 50ms per request.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Step 2 — Data Generation (synthetic Indian loan dataset)
# ─────────────────────────────────────────────────────────────
n = 2000
np.random.seed(42)

age            = np.random.randint(22, 60, n)
income         = np.random.lognormal(11.5, 0.6, n).astype(int)
loan_amount    = np.random.lognormal(12, 0.5, n).astype(int)
credit_score   = np.random.normal(700, 80, n).clip(300, 850).astype(int)
employment_yrs = np.random.exponential(4, n).clip(0, 30).round(1)
num_dependents = np.random.randint(0, 5, n)
city           = np.random.choice(['Mumbai', 'Bangalore', 'Delhi', 'Pune', 'Hyderabad'], n)
existing_loans = np.random.randint(0, 4, n)

# Default probability: low credit score, high loan, many existing loans → higher risk
default_prob = 1 / (1 + np.exp(-(
    -0.01 * credit_score
    + 0.000002 * loan_amount
    - 0.000001 * income
    + 0.3 * existing_loans
    + 3.5
)))
default = (np.random.rand(n) < default_prob).astype(int)

# Inject noise and nulls
income_noisy = income.astype(float)
income_noisy[np.random.choice(n, 80, replace=False)] = np.nan
credit_noisy = credit_score.astype(float)
credit_noisy[np.random.choice(n, 60, replace=False)] = np.nan

df = pd.DataFrame({
    'age': age, 'income': income_noisy, 'loan_amount': loan_amount,
    'credit_score': credit_noisy, 'employment_yrs': employment_yrs,
    'num_dependents': num_dependents, 'city': city,
    'existing_loans': existing_loans, 'default': default
})

print(f'Dataset: {df.shape}  |  Default rate: {df.default.mean():.2%}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Step 3 — Exploratory Data Analysis
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, col in zip(axes.flatten(), ['age', 'income', 'loan_amount', 'credit_score', 'employment_yrs', 'existing_loans']):
    df[df.default==0][col].dropna().hist(ax=ax, alpha=0.5, color='steelblue', label='No Default', bins=25)
    df[df.default==1][col].dropna().hist(ax=ax, alpha=0.5, color='tomato',    label='Default',    bins=25)
    ax.set_title(col)
    ax.legend()

plt.suptitle('Feature Distributions: Default vs No Default', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Step 4 — Preprocessing Pipeline
# ─────────────────────────────────────────────────────────────
df_enc = df.copy()
df_enc['city'] = LabelEncoder().fit_transform(df_enc['city'])

X = df_enc.drop('default', axis=1)
y = df_enc['default']

numeric_features = ['age', 'income', 'loan_amount', 'credit_score',
                    'employment_yrs', 'num_dependents', 'existing_loans', 'city']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.2%}, Test default rate: {y_test.mean():.2%}')

---
## 🟢 Exercise 16.1 — Model Comparison Pipeline *(Level 1 · Est. 15 min)*

> Build full pipelines (preprocessor + classifier) for LR, RF, GBM. Compare using 5-fold CV on AUC-ROC and Recall.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 16.1: Model Comparison
# ─────────────────────────────────────────────────────────────

classifiers = {
    'Logistic Regression':   LogisticRegression(max_iter=1000),
    'Random Forest':         RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, random_state=42),
}
results = []

# YOUR CODE HERE — for each classifier, build Pipeline(preprocessor + clf), run CV

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print('✅ Model comparison complete!')

In [ ]:
# @title ✅ Solution — Exercise 16.1 (click to expand)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, clf in classifiers.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    auc    = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc').mean()
    recall = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='recall').mean()
    f1     = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1').mean()
    results.append({'Model': name, 'AUC-ROC': round(auc, 4), 'Recall': round(recall, 4), 'F1': round(f1, 4)})

results_df = pd.DataFrame(results).sort_values('AUC-ROC', ascending=False)
print(results_df.to_string(index=False))

---
## 🟡 Exercise 16.2 — Threshold Tuning for Business Metric *(Level 2 · Est. 20 min)*

> Train the best model on training set. Plot precision-recall tradeoff vs decision threshold. Find the threshold that achieves Recall ≥ 0.80 while maximising Precision.

In [ ]:
# @title ✅ Solution — Exercise 16.2 (click to expand)
from sklearn.metrics import precision_recall_curve

best_pipe = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', GradientBoostingClassifier(n_estimators=100, random_state=42))])
best_pipe.fit(X_train, y_train)

y_prob = best_pipe.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(9, 5))
plt.plot(thresholds, precision[:-1], label='Precision', color='steelblue')
plt.plot(thresholds, recall[:-1],    label='Recall',    color='darkorange')
plt.axhline(0.80, linestyle='--', color='red', label='Recall target 0.80')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Precision-Recall vs Decision Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Find threshold with recall >= 0.80
valid = [(thresholds[i], precision[i], recall[i])
         for i in range(len(thresholds)) if recall[i] >= 0.80]
if valid:
    best = max(valid, key=lambda x: x[1])
    print(f'Best threshold for Recall>=0.80: {best[0]:.3f}  Precision={best[1]:.3f}  Recall={best[2]:.3f}')
    y_pred_tuned = (y_prob >= best[0]).astype(int)
    print(classification_report(y_test, y_pred_tuned))

---
## 🔴 Exercise 16.3 — Business Impact Report *(Level 3 · Est. 20 min)*

> Assume: avg loan value = ₹5L, loss on default = 40% of loan. Calculate ₹ saved by using ML vs no model. Include cost of false positives (missed business).

In [ ]:
# @title ✅ Solution — Exercise 16.3 (click to expand)

AVG_LOAN        = 500_000   # ₹5L average loan
LOSS_RATE       = 0.40      # 40% recovery failure on default
OPPORTUNITY_COST = 50_000   # ₹50K profit lost per false positive (rejected good customer)

# Default predictions with tuned threshold
if valid:
    threshold = best[0]
else:
    threshold = 0.5
y_pred_final = (y_prob >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm.ravel()

# Baseline: no model, approve all
baseline_loss = (y_test == 1).sum() * AVG_LOAN * LOSS_RATE

# With ML model:
ml_defaults_caught = tp * AVG_LOAN * LOSS_RATE   # money saved by catching defaulters
ml_fp_cost         = fp * OPPORTUNITY_COST        # cost of rejected good customers
ml_net_benefit     = ml_defaults_caught - ml_fp_cost

print('='*55)
print('Business Impact Analysis — Loan Default Model')
print('='*55)
print(f'Test samples: {len(y_test)}')
print(f'Actual defaults: {(y_test==1).sum()}')
print(f'\nConfusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}')
print(f'\nBaseline (no model): Loss = ₹{baseline_loss:,.0f}')
print(f'ML model catches {tp}/{(y_test==1).sum()} defaulters')
print(f'Loss saved by catching defaults: ₹{ml_defaults_caught:,.0f}')
print(f'Cost of false positives:         ₹{ml_fp_cost:,.0f}')
print(f'Net benefit:                     ₹{ml_net_benefit:,.0f}')
print(f'\nROI: {100*ml_net_benefit/baseline_loss:.1f}% of baseline loss recovered')
print('\n✅ This is how you communicate model value to business stakeholders!')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: How do you define success metrics for an ML project?</strong></summary>

**Answer:** Always start with the BUSINESS goal, not the ML metric. Ask: 'What decision does this model inform? What is the cost of a wrong decision?' For a fraud detector, a false negative (missed fraud) costs ₹10,000 but a false positive (legitimate transaction blocked) costs customer satisfaction — these are asymmetric. Translate business costs to a metric: precision/recall tradeoff, custom cost-sensitive metric, or AUC-ROC. Get stakeholder agreement on the target metric BEFORE building anything.
</details>

<details>
<summary><strong>Q2: What is the difference between ColumnTransformer and Pipeline?</strong></summary>

**Answer:** Pipeline chains steps SEQUENTIALLY — each step feeds output to the next. Used for steps that apply to all features (scale → model). ColumnTransformer applies DIFFERENT transformations to DIFFERENT column subsets IN PARALLEL — impute numeric columns one way, encode categorical another way. They combine: Pipeline([('preprocess', ColumnTransformer(...)), ('model', RFC())]) — the ColumnTransformer handles feature-specific preprocessing as one step in the pipeline.
</details>

<details>
<summary><strong>Q3: What should an ML project report include?</strong></summary>

**Answer:** (1) Problem framing: business objective, success metric, constraints. (2) Data summary: sources, shape, quality issues found. (3) Feature engineering decisions and why. (4) Model selection: why this model vs alternatives. (5) Performance: metric on test set with confidence interval. (6) Business impact: translate metric to ₹ or operational impact. (7) Limitations: what the model can't do, edge cases, fairness concerns. (8) Next steps: how to improve, monitoring strategy.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 16 Complete!</h3>
<ul>
<li>End-to-end ML project workflow: framing → data → pipeline → evaluation</li>
<li>ColumnTransformer for mixed feature types</li>
<li>Threshold tuning for business requirements</li>
<li>Business impact quantification in ₹</li>
</ul>
<p><strong>Next:</strong> Chapter 17 — Model Deployment and Production</p>
</div>